# Melanoma metadata cleaning: Full dataset

Before studying the data, we need each row to use the same labels and formats. This notebook cleans the metadata, the information recorded alongside each image, while keeping every row and leaving the source files unchanged. It asks whether we can prepare consistent records without guessing missing information or hiding invalid values.

This begins the full-dataset, or d2, sequence. It follows the same cleaning rules as `../d1/01_data_cleaning.ipynb`, but uses the full source files and writes to a separate folder. It prepares the inputs for `02_data_exploration.ipynb`; it does not replace the subset files.

We keep sex, approximate age and anatomical site available for later models. Missing values receive the label `unknown`. Diagnosis and the benign/malignant description remain in the file for checks, but later models must exclude them because they reveal the answer we want to predict.

## Setup

Set `DATA_DIR` to your local data folder. The code names the source and output files in one place so you can see which files it reads and where it saves the cleaned copies. It also loads the shared label check, so both dataset branches apply the same rule.

In [1]:
from pathlib import Path
import sys

import pandas as pd

# Use this checkout's shared validation checks when run from the project or notebooks folder.
PROJECT_ROOT = next(
    (directory for directory in [Path.cwd(), *Path.cwd().parents]
     if (directory / "src" / "melanoma_detection").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from the project root or its notebooks folder.")
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
from melanoma_detection.metadata_validation import validate_binary_target

# Change this one path to your local data directory.
DATA_DIR = Path(r"C:/Users/thoma/Documents/GitHub/ADS2002-Melanoma/data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

TRAIN_PATH = RAW_DIR / "train.csv"
TEST_PATH = RAW_DIR / "test.csv"
TRAIN_CLEAN_PATH = PROCESSED_DIR / "train_clean.csv"
TEST_CLEAN_PATH = PROCESSED_DIR / "test_clean.csv"

UNKNOWN_LABEL = "unknown"
MODEL_FEATURES = ["sex", "age_approx", "anatom_site_general_challenge"]
LEAKAGE_COLUMNS = ["diagnosis", "benign_malignant", "target"]

print(f"Data directory: {DATA_DIR}")


Data directory: C:\Users\thoma\Documents\GitHub\ADS2002-Melanoma\data


## Load the source metadata

Each row represents an image. `image_name` identifies that image, while `patient_id` links images from the same person. The row and column counts below give us a starting point: cleaning should not remove rows.

The target is the recorded outcome, where `0` means benign and `1` means malignant. We read it as text first so a bad value cannot be rounded into an apparently valid label.

In [2]:
train_raw = pd.read_csv(TRAIN_PATH, dtype={"target": "string"})
test_raw = pd.read_csv(TEST_PATH, dtype={"target": "string"})

print(f"Training rows: {len(train_raw):,}; columns: {train_raw.shape[1]}")
print(f"Test rows: {len(test_raw):,}; columns: {test_raw.shape[1]}")


Training rows: 33,126; columns: 8
Test rows: 10,982; columns: 5


## Check missing information

The table lists only columns with missing values. `missing_count` tells us how many image records lack that field; `missing_percent` divides that count by all rows in that file. A blank table means no missing values were found.

These tables show where cleaning is needed. A graph would add little here because we need the exact counts to check that the missing records are handled consistently.

In [3]:
def missing_profile(frame: pd.DataFrame) -> pd.DataFrame:
    missing_count = frame.isna().sum()
    return pd.DataFrame(
        {
            "missing_count": missing_count,
            "missing_percent": (100 * missing_count / len(frame)).round(3),
        }
    ).query("missing_count > 0")


print("Training missingness:")
print(missing_profile(train_raw).to_string())
print("\nTest missingness:")
print(missing_profile(test_raw).to_string())


Training missingness:
                               missing_count  missing_percent
sex                                       65            0.196
age_approx                                68            0.205
anatom_site_general_challenge            527            1.591

Test missingness:
                               missing_count  missing_percent
anatom_site_general_challenge            351            3.196


## Clean the values without guessing

First, remove extra spaces and treat empty text as missing. Then fill missing sex and site with `unknown`. Keep approximate age as its recorded five-year bucket, such as `45`, and use `unknown` when it is missing. This keeps missing information visible instead of inventing an age from other patients. Later modelling notebooks may choose a numeric age representation within their training data.

Known ages must be numeric, between 0 and 120, and in five-year steps. These are format checks for this dataset, not a claim that every recorded age is correct.

Check the original target before converting it to an integer. Only exact `0` and `1` values are accepted. Converting `1.5` first would turn it into `1` and hide an incorrect label. Reading the target as text also prevents the CSV reader from rounding a value such as `1.000000000000000001` before the check.

In [4]:
EXPECTED_SEX = {"female", "male", UNKNOWN_LABEL}
EXPECTED_SITES = {
    "head/neck",
    "lower extremity",
    "oral/genital",
    "palms/soles",
    "torso",
    "upper extremity",
    UNKNOWN_LABEL,
}


def strip_text_columns(frame: pd.DataFrame) -> pd.DataFrame:
    cleaned = frame.copy()
    for column in cleaned.select_dtypes(include=["object", "string"]).columns:
        values = cleaned[column].astype("string").str.strip()
        cleaned[column] = values.mask(values.eq(""), pd.NA)
    return cleaned


def clean_metadata(frame: pd.DataFrame) -> pd.DataFrame:
    cleaned = strip_text_columns(frame)

    required_columns = {
        "image_name",
        "patient_id",
        "sex",
        "age_approx",
        "anatom_site_general_challenge",
    }
    missing_columns = required_columns - set(cleaned.columns)
    if missing_columns:
        raise ValueError(f"Required columns are missing: {sorted(missing_columns)}")

    for column in ["sex", "anatom_site_general_challenge"]:
        cleaned[column] = cleaned[column].fillna(UNKNOWN_LABEL)

    age_numeric = pd.to_numeric(cleaned["age_approx"], errors="coerce")
    malformed_age = cleaned["age_approx"].notna() & age_numeric.isna()
    if malformed_age.any():
        bad_values = sorted(cleaned.loc[malformed_age, "age_approx"].unique())
        raise ValueError(f"Non-numeric age values found: {bad_values}")

    known_age = age_numeric.dropna()
    if not known_age.between(0, 120).all():
        raise ValueError("age_approx contains values outside the expected 0-120 range.")
    if not known_age.mod(5).eq(0).all():
        raise ValueError("age_approx contains values outside the expected five-year buckets.")

    cleaned["age_approx"] = age_numeric.map(
        lambda value: str(int(value)) if pd.notna(value) else UNKNOWN_LABEL
    )

    if "target" in cleaned.columns:
        cleaned["target"] = validate_binary_target(cleaned["target"])

    return cleaned


train_clean = clean_metadata(train_raw)
test_clean = clean_metadata(test_raw)


## Check the cleaned records

The checks below confirm that row counts stay the same, image names are unique, patient identifiers are present, and the feature values use the expected labels. They also check that the target agrees with the benign/malignant description.

“All validation checks passed” means these checks found no problem. If a check fails, fix the source issue before using the outputs.

In [5]:
def validate_common(frame: pd.DataFrame, source: pd.DataFrame, name: str) -> None:
    assert len(frame) == len(source), f"{name}: row count changed during cleaning."
    assert frame["image_name"].notna().all(), f"{name}: missing image_name."
    assert frame["patient_id"].notna().all(), f"{name}: missing patient_id."
    assert frame["image_name"].is_unique, f"{name}: duplicate image_name values."
    assert set(frame["sex"].unique()) <= EXPECTED_SEX, f"{name}: unexpected sex value."
    assert set(frame["anatom_site_general_challenge"].unique()) <= EXPECTED_SITES, (
        f"{name}: unexpected anatomical-site value."
    )
    assert frame[MODEL_FEATURES].isna().sum().sum() == 0, (
        f"{name}: model features still contain missing values."
    )


validate_common(train_clean, train_raw, "train")
validate_common(test_clean, test_raw, "test")

assert set(train_clean["target"].unique()) <= {0, 1}, "Unexpected target value."
assert (
    ((train_clean["target"] == 1) & (train_clean["benign_malignant"] != "malignant"))
    | ((train_clean["target"] == 0) & (train_clean["benign_malignant"] != "benign"))
).sum() == 0, "target and benign_malignant disagree."

print("All validation checks passed.")


All validation checks passed.


### Read the change table

Compare `missing_before` with `unknown_after`: missing information should remain visible as an unknown category. `missing_after` should be zero for the three model fields. Filling a missing value changes how we store it; it does not recover the information.

Each row of this table describes one field in one file. Use it to check the cleaning rule, rather than to judge model quality.

In [6]:
rows = []
for dataset_name, source, cleaned in [
    ("train", train_raw, train_clean),
    ("test", test_raw, test_clean),
]:
    for column in MODEL_FEATURES:
        rows.append(
            {
                "dataset": dataset_name,
                "column": column,
                "missing_before": int(source[column].isna().sum()),
                "unknown_after": int(cleaned[column].eq(UNKNOWN_LABEL).sum()),
                "missing_after": int(cleaned[column].isna().sum()),
            }
        )

cleaning_summary = pd.DataFrame(rows)
print(cleaning_summary.to_string(index=False))


dataset                        column  missing_before  unknown_after  missing_after
  train                           sex              65             65              0
  train                    age_approx              68             68              0
  train anatom_site_general_challenge             527            527              0
   test                           sex               0              0              0
   test                    age_approx               0              0              0
   test anatom_site_general_challenge             351            351              0


## Save the cleaned copies

The outputs go to `data/processed`. The original files remain available, so we can trace the cleaned records back to their source. Running this cell again writes the cleaned copies to the same output paths.

In [7]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
train_clean.to_csv(TRAIN_CLEAN_PATH, index=False)
test_clean.to_csv(TEST_CLEAN_PATH, index=False)

for path in [TRAIN_CLEAN_PATH, TEST_CLEAN_PATH]:
    print(f"Wrote: {path}")


Wrote: C:\Users\thoma\Documents\GitHub\ADS2002-Melanoma\data\processed\train_clean.csv
Wrote: C:\Users\thoma\Documents\GitHub\ADS2002-Melanoma\data\processed\test_clean.csv


## Limitations

These checks can find missing fields, unexpected formats and some conflicting labels. They cannot confirm that the original age, sex, site or diagnosis was recorded correctly. An unknown category preserves a gap in the records, but cannot explain why the information is missing or remove any bias caused by that gap.

Five-year age buckets lose detail within each bucket. The checks also do not inspect image contents, identify visually similar images, or establish how well a future model will work. No model is trained in this notebook, and the preparation checks do not provide a test-set performance result.

## Summary

The saved output keeps 33,126 training rows and 10,982 test rows. In training, 65 sex values, 68 ages and 527 sites become `unknown`; in the test file, 351 sites become `unknown`. The saved validation checks pass.

Retain this preparation method: it preserves rows, makes missing information explicit and stops on invalid labels. There is no model to retain at this stage. Next, open `02_data_exploration.ipynb` to study the cleaned training records and see why images from the same patient need to stay together during evaluation.